# 01 — Native tool-calling baseline

## Goal

Run the upstream BF16 model through the exact six-tool schema,
preserve typed failures and measure episode cost before SFT.

This notebook implements the documented `trusted-dev` fallback
for reviewed pilot repositories. It is **not** a security
boundary for arbitrary public code. Replace the executor with a
Harbor isolated backend before scaling collection or RL.

In [ ]:
import subprocess
import sys
from pathlib import Path

# The Git pins supply current Unsloth/Qwen3.8 support. Transformers, TRL and
# Datasets deliberately use the mutually compatible versions from the adjacent
# official Unsloth Qwen3.5 27B notebook. Do not replace these with branch-head
# SHAs without resolving package metadata together first.
GIT_REVISIONS = {
    "unsloth": "c87fe20e32aca9ceb2dc5059c2987738f32446e8",
    "unsloth_zoo": "5b239e574f03ab3077c17e49aeef3cacfe7cdd4e",
}

import torch

torch_version = torch.__version__.split("+", 1)[0]
torch_minor = ".".join(torch_version.split(".")[:2])
torchao_by_torch = {"2.8": "0.16.0", "2.9": "0.16.0", "2.10": "0.16.0", "2.11": "0.18.0"}
xformers_by_torch = {"2.8": "0.0.32.post2", "2.9": "0.0.33.post1", "2.10": "0.0.34", "2.11": "0.0.34"}
if torch_minor not in torchao_by_torch:
    raise RuntimeError(
        f"No reviewed Colab dependency set for torch {torch.__version__}. "
        f"Expected one of {sorted(torchao_by_torch)}; update the compatibility matrix first."
    )

COMPATIBILITY_PINS = {
    "transformers": "5.3.0",
    "trl": "0.22.2",
    "datasets": "4.3.0",
    "peft": "0.19.0",
    "torchao": torchao_by_torch[torch_minor],
    "xformers": xformers_by_torch[torch_minor],
}
INSTALLER_REVISION = "colab-v2"
pin_key = "-".join(value.replace(".", "") for value in COMPATIBILITY_PINS.values())
git_key = "-".join(value[:8] for value in GIT_REVISIONS.values())
INSTALL_KEY = f"{INSTALLER_REVISION}-torch{torch_minor}-{git_key}-{pin_key}"
INSTALL_MARKER = Path(f"/content/.qwen38_env_{INSTALL_KEY}")
PIP_LOG = Path("/content/qwen38_pip_install.log")
FORCE_INSTALL = False

def install_phase(name: str, packages: list[str], *, no_deps: bool = False) -> None:
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        "--no-cache-dir",
        "--log",
        str(PIP_LOG),
    ]
    if no_deps:
        command.append("--no-deps")
    command.extend(packages)
    print(f"\n=== install phase: {name} ===")
    print("\n".join(f"  {package}" for package in packages))
    result = subprocess.run(command, check=False)
    if result.returncode:
        log_tail = (
            "\n".join(PIP_LOG.read_text(errors="replace").splitlines()[-120:])
            if PIP_LOG.exists()
            else "[pip did not create its log file]"
        )
        print(f"\n--- tail of {PIP_LOG} ---\n{log_tail}")
        raise RuntimeError(
            f"Package installation failed during {name!r} with exit code {result.returncode}. "
            f"The detailed log is at {PIP_LOG}."
        )

if FORCE_INSTALL or not INSTALL_MARKER.exists():
    if PIP_LOG.exists():
        PIP_LOG.unlink()
    install_phase("packaging tools", ["pip", "setuptools==80.9.0", "wheel>=0.42.0"])
    install_phase("Qwen3.8 training stack", [
        f"unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git@{GIT_REVISIONS['unsloth_zoo']}",
        f"unsloth @ git+https://github.com/unslothai/unsloth.git@{GIT_REVISIONS['unsloth']}",
        f"torch=={torch_version}",
        f"torchao=={COMPATIBILITY_PINS['torchao']}",
        f"transformers=={COMPATIBILITY_PINS['transformers']}",
        f"trl=={COMPATIBILITY_PINS['trl']}",
        f"datasets=={COMPATIBILITY_PINS['datasets']}",
        f"peft=={COMPATIBILITY_PINS['peft']}",
        "accelerate",
        "bitsandbytes",
        "trackio",
        "huggingface_hub>=0.34.0,<2.0",
        "hf_transfer",
        "sentencepiece>=0.2.0",
        "protobuf",
        "pytest",
        "jmespath",
    ])
    install_phase(
        "PyTorch-matched xFormers wheel",
        [f"xformers=={COMPATIBILITY_PINS['xformers']}"],
        no_deps=True,
    )
    INSTALL_MARKER.write_text(INSTALL_KEY)
    print("Packages installed. Restart the Colab runtime, then rerun this notebook from the top.")
else:
    print(f"Pinned environment already installed: {INSTALL_KEY}")

In [ ]:
import json
import os
import platform
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import torch
from huggingface_hub import login, whoami

if "GIT_REVISIONS" not in globals():
    raise RuntimeError(
        "This runtime was restarted. Rerun the notebook from the first cell; "
        "the install marker will skip the expensive package installation."
    )
if "COMPATIBILITY_PINS" not in globals():
    raise RuntimeError("Missing compatibility pins; rerun the notebook from the first cell.")

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab G4 GPU runtime before continuing.")

gpu = torch.cuda.get_device_properties(0)
gpu_total_gib = gpu.total_memory / 1024**3
# A vendor-labelled 96 GB card can be reported as about 89.4 GiB because
# PyTorch converts the byte count with a binary divisor. Keep the floor well
# above the roughly 44.7 GiB reported for a 48 GB card without rejecting G4.
MIN_G4_TOTAL_GIB = 85.0
print(
    f"GPU: {gpu.name} ({gpu_total_gib:.1f} GiB total), "
    f"capability={torch.cuda.get_device_capability(0)}"
)
if gpu_total_gib < MIN_G4_TOTAL_GIB:
    raise RuntimeError(
        "This suite expects the nominal 96 GB Colab G4 runtime. "
        f"PyTorch reports {gpu_total_gib:.1f} GiB total; expected at least "
        f"{MIN_G4_TOTAL_GIB:.0f} GiB. A value near 45 GiB usually indicates "
        "the 48 GB GPU variant."
    )

hf_token = userdata.get("HF_TOKEN") if userdata is not None else os.getenv("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add a write-capable HF_TOKEN to Colab Secrets before continuing.")
login(token=hf_token, add_to_git_credential=False)
HF_USERNAME = whoami()["name"]

def package_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "missing"

observed_pins = {name: package_version(name) for name in COMPATIBILITY_PINS}
pin_mismatches = {
    name: {"expected": expected, "observed": observed_pins[name]}
    for name, expected in COMPATIBILITY_PINS.items()
    if observed_pins[name] != expected
}
if pin_mismatches:
    raise RuntimeError(
        "The runtime does not match the reviewed compatibility set. "
        f"Rerun the install cell with FORCE_INSTALL=True: {pin_mismatches}"
    )

RUN_ROOT = Path("/content/qwen38_runs")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
runtime_manifest = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": gpu.name,
    "gpu_total_gib": round(gpu_total_gib, 2),
    "packages": {
        name: package_version(name)
        for name in ["unsloth", "unsloth_zoo", "transformers", "trl", "peft", "datasets"]
    },
    "git_revisions": GIT_REVISIONS,
    "compatibility_pins": COMPATIBILITY_PINS,
}
(RUN_ROOT / "runtime_manifest.json").write_text(json.dumps(runtime_manifest, indent=2))
print(json.dumps(runtime_manifest, indent=2))
print(f"Authenticated as {HF_USERNAME}")

## Parameters

In [ ]:
from dataclasses import dataclass, replace
from datetime import datetime, timezone
import hashlib
import re
import shutil
import subprocess
import tempfile
import time
import uuid

from unsloth import FastLanguageModel

MODEL_ID = "unsloth/Qwen3.8-27B"
MAX_SEQUENCE_LENGTH = 16384
MAX_NEW_TOKENS_PER_TURN = 1024
MAX_TOOL_CALLS = 10
EPISODE_TIMEOUT_SECONDS = 480
BASELINE_SEEDS = (3407, 9176, 20261)
DEMO_MODE = True
PILOT_MANIFEST = Path("/content/pilot_tasks.jsonl")
RESULTS_DIR = RUN_ROOT / "baseline"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
secret_markers = ("TOKEN", "SECRET", "PASSWORD", "CREDENTIAL")
TASK_ENV = {
    key: value for key, value in os.environ.items()
    if not any(marker in key.upper() for marker in secret_markers)
}

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    load_in_4bit=False,
    full_finetuning=False,
)
FastLanguageModel.for_inference(model)

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List files below a repository-relative directory.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a UTF-8 repository file with bounded output.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search repository text using a regular expression.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apply_patch",
            "description": "Apply a unified diff to files inside the repository.",
            "parameters": {
                "type": "object",
                "properties": {"patch": {"type": "string"}},
                "required": ["patch"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_tests",
            "description": "Run an allow-listed repository test profile.",
            "parameters": {
                "type": "object",
                "properties": {"profile": {"type": "string", "enum": ["unit"]}},
                "required": ["profile"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "shell",
            "description": "Run a restricted allow-listed command. It is disabled in the pilot.",
            "parameters": {
                "type": "object",
                "properties": {"command": {"type": "string"}},
                "required": ["command"],
                "additionalProperties": False,
            },
        },
    },
]

def _without_arrow_nulls(value):
    """Remove null struct fields inserted by a Datasets/Arrow round trip."""
    if isinstance(value, dict):
        cleaned = {}
        for key, item in value.items():
            normalized = _without_arrow_nulls(item)
            if normalized is not None:
                cleaned[key] = normalized
        return cleaned
    if isinstance(value, list):
        return [_without_arrow_nulls(item) for item in value]
    return value

def canonical_tool_schema(tools: list[dict]) -> str:
    """Return a stable semantic fingerprint while retaining tool order."""
    return json.dumps(
        _without_arrow_nulls(tools),
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    )

TOOL_SCHEMA_JSON = canonical_tool_schema(TOOLS)

def rendered_tool_schema(rendered_prompt: str) -> str:
    """Extract and canonicalise JSON tool declarations from a Qwen prompt."""
    start_tag = "<tools>"
    end_tag = "</tools>"
    if start_tag not in rendered_prompt or end_tag not in rendered_prompt:
        raise ValueError("Rendered prompt does not contain a <tools> block.")
    payload = rendered_prompt.split(start_tag, 1)[1].split(end_tag, 1)[0]
    try:
        rendered_tools = [
            json.loads(line)
            for line in payload.splitlines()
            if line.strip()
        ]
    except json.JSONDecodeError as exc:
        raise ValueError("Rendered <tools> block is not newline-delimited JSON.") from exc
    return canonical_tool_schema(rendered_tools)

def canonical_to_qwen(messages: list[dict]) -> list[dict]:
    """Fold an initial developer message into system for the HF tokenizer.

    The adapter performs the same mapping in training and deployment. The
    official safetensor tokenizer currently accepts system/user/assistant/tool.
    """
    converted = []
    pending_system = []
    for stored_message in messages:
        message = _without_arrow_nulls(stored_message)
        role = message["role"]
        if role in {"system", "developer"} and not converted:
            pending_system.append(str(message.get("content", "")))
            continue
        if pending_system:
            converted.append({"role": "system", "content": "\n\n".join(pending_system)})
            pending_system = []
        converted.append(message)
    if pending_system:
        converted.append({"role": "system", "content": "\n\n".join(pending_system)})
    return converted

def render_chat(messages: list[dict], *, add_generation_prompt: bool) -> str:
    return tokenizer.apply_chat_template(
        canonical_to_qwen(messages),
        tools=TOOLS,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
        enable_thinking=True,
        reasoning_effort="medium",
        preserve_thinking=True,
    )

## Parse Qwen3.8's native XML tool calls

In [ ]:
TOOL_CALL_RE = re.compile(
    r"<tool_call>\s*<function=([^>\n]+)>\s*(.*?)</function>\s*</tool_call>",
    re.DOTALL,
)
PARAM_RE = re.compile(
    r"<parameter=([^>\n]+)>(?:\r?\n)?(.*?)\s*</parameter>",
    re.DOTALL,
)

def split_reasoning(text: str) -> tuple[str, str]:
    if "</think>" in text:
        reasoning, content = text.split("</think>", 1)
        return reasoning.removeprefix("<think>").strip(), content.strip()
    return "", text.strip()

def parse_tool_calls(text: str) -> tuple[str, list[dict]]:
    reasoning, content = split_reasoning(text)
    calls = []
    for function_name, body in TOOL_CALL_RE.findall(content):
        arguments = {
            name.strip(): (
                value.rstrip("\r\n")
                if name.strip() == "patch"
                else value.strip()
            )
            for name, value in PARAM_RE.findall(body)
        }
        calls.append({
            "type": "function",
            "function": {"name": function_name.strip(), "arguments": arguments},
        })
    return reasoning, calls

parser_probe = ("</think>\n\n<tool_call>\n<function=read_file>\n"
                "<parameter=path>\nsrc/cache.py\n</parameter>\n"
                "</function>\n</tool_call>")
assert parse_tool_calls(parser_probe)[1][0]["function"]["arguments"]["path"] == "src/cache.py"

## Trusted pilot task and executor

In [ ]:
@dataclass(frozen=True)
class PilotTask:
    task_id: str
    repo_path: str
    request: str
    visible_test_command: list[str]
    hidden_test_command: list[str]

def make_demo_task() -> PilotTask:
    repo = Path("/content/qwen38_demo_repo")
    if repo.exists():
        shutil.rmtree(repo)
    (repo / "src").mkdir(parents=True)
    (repo / "tests").mkdir()
    (repo / "src" / "clamp.py").write_text(
        "def clamp(value, lower, upper):\n"
        "    return min(lower, max(upper, value))\n"
    )
    (repo / "tests" / "test_clamp.py").write_text(
        "from src.clamp import clamp\n\n"
        "def test_value_in_range():\n"
        "    assert clamp(5, 0, 10) == 5\n"
    )
    hidden = Path("/content/qwen38_hidden_test.py")
    hidden.write_text(
        "from pathlib import Path\n"
        "ns = {}\n"
        "exec((Path.cwd() / 'src' / 'clamp.py').read_text(), ns)\n"
        "clamp = ns['clamp']\n"
        "assert clamp(-2, 0, 10) == 0\n"
        "assert clamp(20, 0, 10) == 10\n"
        "assert clamp(5, 0, 10) == 5\n"
    )
    subprocess.run(["git", "init", "-q"], cwd=repo, check=True)
    subprocess.run(["git", "add", "."], cwd=repo, check=True)
    subprocess.run(
        ["git", "-c", "user.name=pilot", "-c", "user.email=pilot@example.invalid", "commit", "-qm", "fixture"],
        cwd=repo,
        check=True,
    )
    return PilotTask(
        task_id="demo/clamp-001",
        repo_path=str(repo),
        request="Fix clamp so values inside the range are unchanged and out-of-range values use the nearest bound. Run the unit tests.",
        visible_test_command=[sys.executable, "-m", "pytest", "-q"],
        hidden_test_command=[sys.executable, str(hidden)],
    )

def load_tasks() -> list[PilotTask]:
    if DEMO_MODE:
        return [make_demo_task()]
    if not PILOT_MANIFEST.exists():
        raise FileNotFoundError(PILOT_MANIFEST)
    return [PilotTask(**json.loads(line)) for line in PILOT_MANIFEST.read_text().splitlines() if line.strip()]

def rooted(root: Path, relative: str) -> Path:
    candidate = (root / relative).resolve()
    if candidate != root and root not in candidate.parents:
        raise ValueError("path escapes repository root")
    return candidate

def execute_tool(task: PilotTask, name: str, arguments: dict) -> str:
    root = Path(task.repo_path).resolve()
    if name == "list_files":
        base = rooted(root, arguments["path"])
        files = [str(path.relative_to(root)) for path in base.rglob("*") if path.is_file() and ".git" not in path.parts]
        return "\n".join(files[:200]) or "[no files]"
    if name == "read_file":
        return rooted(root, arguments["path"]).read_text(errors="replace")[:20000]
    if name == "search":
        try:
            regex = re.compile(arguments["query"])
        except re.error as exc:
            return f"invalid regular expression: {exc}"
        hits = []
        scanned_bytes = 0
        for path in sorted(root.rglob("*")):
            if not path.is_file() or ".git" in path.parts:
                continue
            try:
                payload = path.read_bytes()
            except OSError as exc:
                hits.append(f"{path.relative_to(root)}:read_error:{exc}")
                continue
            if b"\x00" in payload:
                continue
            scanned_bytes += len(payload)
            if scanned_bytes > 5_000_000:
                hits.append("[search truncated after 5 MB]")
                break
            for line_no, line in enumerate(payload.decode("utf-8", errors="replace").splitlines(), 1):
                if regex.search(line):
                    hits.append(f"{path.relative_to(root)}:{line_no}:{line}")
                    if len(hits) >= 200:
                        hits.append("[search truncated after 200 matches]")
                        return "\n".join(hits)[:20000]
        return "\n".join(hits)[:20000] or "[no matches]"
    if name == "apply_patch":
        result = subprocess.run(
            ["git", "apply", "--whitespace=nowarn", "-"],
            cwd=root,
            input=arguments["patch"],
            text=True,
            capture_output=True,
            timeout=30,
        )
        return "patch applied" if result.returncode == 0 else f"patch rejected: {result.stderr[:4000]}"
    if name == "run_tests":
        if arguments["profile"] != "unit":
            return "unknown test profile"
        result = subprocess.run(
            task.visible_test_command,
            cwd=root,
            env=TASK_ENV,
            text=True,
            capture_output=True,
            timeout=120,
        )
        return f"exit={result.returncode}\n{(result.stdout + result.stderr)[-12000:]}"
    if name == "shell":
        return "shell is disabled in the pilot; use the semantic tools"
    return f"unknown tool: {name}"

## Run a bounded episode

In [ ]:
def generate_turn(messages: list[dict]) -> tuple[str, int, int]:
    rendered = render_chat(messages, add_generation_prompt=True)
    inputs = tokenizer(
        text=rendered,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS_PER_TURN,
            temperature=1.0,
            top_p=0.95,
            top_k=20,
            do_sample=True,
            use_cache=True,
        )
    new_ids = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=False), inputs["input_ids"].numel(), new_ids.numel()

def run_episode(task: PilotTask, seed: int) -> dict:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    start = time.monotonic()
    messages = [
        {"role": "developer", "content": "Work only in the provided repository. Inspect before editing and run tests before completing."},
        {"role": "user", "content": task.request},
    ]
    prompt_tokens = completion_tokens = tool_count = 0
    termination = "assistant_complete"
    final_text = ""

    for _ in range(MAX_TOOL_CALLS + 1):
        if time.monotonic() - start > EPISODE_TIMEOUT_SECONDS:
            termination = "timeout"
            break
        raw, prompt_count, completion_count = generate_turn(messages)
        prompt_tokens += prompt_count
        completion_tokens += completion_count
        reasoning, calls = parse_tool_calls(raw)
        if not calls:
            _, final_text = split_reasoning(raw)
            messages.append({"role": "assistant", "reasoning_content": reasoning, "content": final_text})
            break
        if tool_count + len(calls) > MAX_TOOL_CALLS:
            termination = "tool_budget"
            break
        messages.append({"role": "assistant", "reasoning_content": reasoning, "content": "", "tool_calls": calls})
        for call in calls:
            function = call["function"]
            try:
                observation = execute_tool(task, function["name"], function["arguments"])
            except Exception as exc:
                observation = f"tool_error: {type(exc).__name__}: {exc}"
            messages.append({"role": "tool", "name": function["name"], "content": observation})
            tool_count += 1
    else:
        termination = "tool_budget"

    hidden = subprocess.run(
        task.hidden_test_command,
        cwd=task.repo_path,
        env=TASK_ENV,
        text=True,
        capture_output=True,
        timeout=120,
    )
    elapsed = time.monotonic() - start
    return {
        "trajectory_id": str(uuid.uuid4()),
        "task_id": task.task_id,
        "seed": seed,
        "model_id": MODEL_ID,
        "adapter_version": "qwen-native-tools-v0.1",
        "messages": messages,
        "termination": termination,
        "success": hidden.returncode == 0,
        "hidden_output": (hidden.stdout + hidden.stderr)[-4000:],
        "usage": {
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "tool_calls": tool_count,
            "wall_seconds": round(elapsed, 3),
        },
        "final_text": final_text,
    }

def run_isolated_episode(task: PilotTask, seed: int) -> dict:
    attempt_root = Path(tempfile.mkdtemp(prefix="qwen38_baseline_"))
    working_repo = attempt_root / "repo"
    shutil.copytree(task.repo_path, working_repo)
    attempt = replace(task, repo_path=str(working_repo))
    try:
        return run_episode(attempt, seed)
    finally:
        shutil.rmtree(attempt_root, ignore_errors=True)

tasks = load_tasks()
active_seeds = BASELINE_SEEDS[:1] if DEMO_MODE else BASELINE_SEEDS
trajectories = [
    run_isolated_episode(task, seed)
    for task in tasks
    for seed in active_seeds
]
print(json.dumps([{k: v for k, v in row.items() if k != "messages"} for row in trajectories], indent=2))

## Persist results and estimate the next gate

In [ ]:
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
trace_path = RESULTS_DIR / f"trajectories-{timestamp}.jsonl"
trace_path.write_text("\n".join(json.dumps(row) for row in trajectories) + "\n")

durations = [row["usage"]["wall_seconds"] for row in trajectories]
mean_seconds = sum(durations) / len(durations)
projected_candidate_hours = 24 * 3 * mean_seconds / 3600
summary = {
    "unique_tasks": len(tasks),
    "attempts": len(trajectories),
    "seeds": list(active_seeds),
    "successes": sum(row["success"] for row in trajectories),
    "mean_episode_seconds": mean_seconds,
    "candidate_gate_gpu_hours_at_observed_mean": projected_candidate_hours,
    "trace_path": str(trace_path),
}
(RESULTS_DIR / f"summary-{timestamp}.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

# Upload after manual trace review. This guards against publishing raw reasoning accidentally.
PUSH_PRIVATE_RESULTS = False
if PUSH_PRIVATE_RESULTS:
    from huggingface_hub import HfApi
    HfApi().upload_folder(
        repo_id=f"{HF_USERNAME}/qwen38-code-pilot-results",
        repo_type="dataset",
        folder_path=str(RESULTS_DIR),
        private=True,
    )

## Checks and next step

Manually inspect every pilot trace. Infrastructure errors must
be separated from model failures. Replace demo mode with the
frozen 12-task manifest, then use the resulting failure mix to
decide which native trajectories to collect for SFT.